In [251]:
import pandas as pd
import numpy as np

Obsługa **indeksowania hierarchicznego** jest ważnym elementem biblioteki pandas umożliwiającym
przypisanie do jednej osi wielu poziomów indeksowania (przypisanie dwóch lub więcej indeksów).

In [252]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [253]:
data

a  1    0.921729
   2    0.342513
   3    0.051418
b  1    0.758806
   3    0.634544
c  1    0.576145
   2    0.516056
d  2    0.724921
   3    0.278693
dtype: float64

Wyświetloną czytelnie serią, której indeksem jest obiekt MultiIndex.

In [254]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

Obiekty o indeksie hierarchicznym obsługują tzw. **indeksowanie częściowe**. Indeksowanie to umożliwia zwięzły wybór podzbioru danych

In [255]:
data["b"]

,0
1,0.758806
3,0.634544


In [256]:
data["b":"c"]

b  1    0.758806
   3    0.634544
c  1    0.576145
   2    0.516056
dtype: float64

In [257]:
data.loc[["b", "d"]]

b  1    0.758806
   3    0.634544
d  2    0.724921
   3    0.278693
dtype: float64

## Przykład na danych

In [258]:
# Dane sprzedażowe w dwóch miastach, w dwóch latach
df = pd.DataFrame({
    "miasto":   ["Łódź", "Łódź", "Kraków", "Kraków"],
    "rok":      [2023,   2024,   2023,     2024],
    "sprzedaz": [120,    145,    200,      230]
})
df

,miasto,rok,sprzedaz
0,Łódź,2023,120
1,Łódź,2024,145
2,Kraków,2023,200
3,Kraków,2024,230


In [259]:
df.index

RangeIndex(start=0, stop=4, step=1)

### Zwykły filtrowanie

Za każdym razem: filtr po mieście, filtr po roku, wybór kolumny. Trzy operacje dla jednej wartości.

In [260]:
# Sprzedaż w Łodzi w 2024 - działa, ale rozwlekle:
df[(df.miasto == "Łódź") & (df.rok == 2024)]["sprzedaz"]

,sprzedaz
1,145


In [261]:
# Sprzedaż w Krakowie w 2023 - znowu to samo:
df[(df.miasto == "Kraków") & (df.rok == 2023)]["sprzedaz"]

,sprzedaz
2,200


### MultiIndex - indeks jako "ścieżka"

In [262]:
# Ten sam zbiór, ale z hierarchicznym indeksem:
sprzedaz = pd.Series(
    [120, 145, 200, 230],
    index=[["Łódź", "Łódź", "Kraków", "Kraków"],
           [2023,   2024,   2023,     2024]]
)
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [263]:
# Sam indeks to obiekt MultiIndex:
sprzedaz.index

MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           )

### Indeksowanie - podstawowe wzorce

In [264]:
# Pojedyncza wartość - krotka (miasto, rok):
sprzedaz["Łódź", 2024]

np.int64(145)

In [265]:
# Wszystko dla Łodzi (wybór zewnętrznego poziomu):
sprzedaz["Łódź"]

,0
2023,120
2024,145


In [266]:
# Wszystkie miasta, ale tylko rok 2023 (wewnętrzny poziom):
sprzedaz.loc[:, 2023]

,0
Łódź,120
Kraków,200


In [267]:
# Zakres miast (uwaga: wymaga posortowanego indeksu):
sprzedaz.sort_index().loc["Kraków":"Łódź"]

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64

## Z Series do DataFrame i z powrotem

In [268]:
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [269]:
# unstack - "wypchnięcie" wewnętrznego poziomu do kolumn:
tabela = sprzedaz.unstack()
print(tabela)
print(type(tabela))

        2023  2024
Kraków   200   230
Łódź     120   145
<class 'pandas.core.frame.DataFrame'>


In [270]:
# stack - operacja odwrotna: kolumny wracają do indeksu:
tab_stack = tabela.stack()
print(tab_stack)
print(type(tab_stack))

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64
<class 'pandas.core.series.Series'>


In [271]:
print(df)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


In [272]:
# Konwersja płaskiej ramki na hierarchiczną - set_index:
df_hier = df.set_index(["miasto", "rok"])
print(df_hier)
print(df_hier.index)

             sprzedaz
miasto rok           
Łódź   2023       120
       2024       145
Kraków 2023       200
       2024       230
MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           names=['miasto', 'rok'])


In [273]:
# I z powrotem - reset_index:
df2 = df_hier.reset_index()
print(df2)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


### Tabele przestawne (ang. _Pivot Table_)

In [274]:
# Przekształcamy płaską ramkę w tabelę przestawną
df2_my_pivot = df.set_index(["miasto", "rok"])["sprzedaz"].unstack()
print(df2_my_pivot)

rok     2023  2024
miasto            
Kraków   200   230
Łódź     120   145


In [275]:
df_pivot = pd.pivot_table(df,
                         index="miasto",
                         columns="rok",
                         values="sprzedaz")
print(df_pivot)

rok      2023   2024
miasto              
Kraków  200.0  230.0
Łódź    120.0  145.0


In [276]:
df_kwartaly = pd.DataFrame({
    "miasto":   ["Łódź"]*4 + ["Kraków"]*4,
    "rok":      [2023]*4 + [2023]*4,
    "kwartal":  ["Q1","Q2","Q3","Q4"] * 2,
    "sprzedaz": [30, 25, 35, 30, 50, 45, 55, 50]
})

df_kwartaly_sum = pd.pivot_table(df_kwartaly,
                                index="miasto",
                                columns="rok",
                                values="sprzedaz",
                                aggfunc="sum")
print(df_kwartaly_sum)

rok     2023
miasto      
Kraków   200
Łódź     120


### Agregacje po poziomie

In [277]:
sprzedaz.index.names = ["miasto", "rok"]
sprzedaz.groupby(level="miasto").sum()

,0
miasto,
Kraków,430
Łódź,265


In [278]:
sprzedaz.groupby(level="rok").mean()

,0
rok,
2023,160.0
2024,187.5


### Agregacja w DataFrame z MultiIndex

In [279]:
frame = pd.DataFrame({
    "sprzedaz": [120, 145, 200, 230],
    "koszty":   [80,  90,  130, 140]
}, index=[["Łódź","Łódź","Kraków","Kraków"],
          [2023,  2024,  2023,    2024]])
frame.index.names = ["miasto", "rok"]
frame.groupby(level="miasto").sum()

,sprzedaz,koszty
miasto,,
Kraków,430,270
Łódź,265,170


In [280]:
frame.groupby(level="miasto").agg(["sum", "mean"])

sprzedaz        koszty       
            sum   mean    sum   mean
miasto                              
Kraków      430  215.0    270  135.0
Łódź        265  132.5    170   85.0

In [281]:
frame["sprzedaz"].groupby(level="miasto").agg(
    total="sum", srednia="mean", rozstep=lambda x: x.max()-x.min()
    )

,total,srednia,rozstep
miasto,,,
Kraków,430,215.0,30
Łódź,265,132.5,25


# *Zadania*

Sieć "ModaŁódź" ma sklepy w trzech miastach. Każdy sklep sprzedaje trzy kategorie produktów. Dane obejmują 4 kwartały 2023 i 2024 roku.

## **Zadanie 1 — Budowanie MultiIndex**

a) Utwórz ramkę `df_hi` z hierarchicznym indeksem (`miasto`, `kategoria`, `rok`, `kwartal`). Posortuj indeks.

b) Wyświetl liczbę poziomów indeksu i ich nazwy.

c) Ile unikalnych kombinacji indeksu istnieje? Użyj `.index` do odpowiedzi.

In [282]:
df = pd.read_csv("sklepy_moda.csv")
df_hi = df.set_index(["miasto", "kategoria", "rok", "kwartal"]).sort_index()
print("Posortowany df_hi:")
print(df_hi)
print("\nLiczba poziomow indeksu:")
print(df_hi.index.nlevels)
print("\nNazwy poziomow indeksu:")
print(df_hi.index.names)
print("\nIlosc unikalnych kombinacji indeksu:")
print(df_hi.index.size)

Posortowany df_hi:
                               Unnamed: 0  przychod_tys  koszt_tys  sztuki
miasto kategoria rok  kwartal                                             
Kraków Damska    2023 Q1               24          71.3       51.9     806
                      Q2               25          90.8       52.1    1179
                      Q3               26          77.4       54.5     886
                      Q4               27         116.9       76.5    1285
                 2024 Q1               28          73.0       44.7     843
...                                   ...           ...        ...     ...
Łódź   Męska     2023 Q4               11          63.5       36.5     728
                 2024 Q1               12          47.3       29.0     550
                      Q2               13          47.9       31.6     445
                      Q3               14          52.3       29.7     490
                      Q4               15          71.1       39.7     730

[72 r

## **Zadanie 2 — Selekcje na MultiIndex**

a) Wybierz wszystkie dane dla Łodzi.
    
b) Wybierz dane dla Łodzi, kategorii Damska.
    
c) Wybierz dane dla wszystkich miast, ale tylko rok 2024. (Wskazówka: użyj metodę `xs()`)

d) Wybierz Q4 z obu lat, ale tylko dla Krakowa i kategorii Męska.

In [287]:
print("\n Wszystkie dane dla Łodzi")
print(df_hi.loc["Łódź"])

print("\n Łódź, kategoria Damska")
print(df_hi.loc[("Łódź", "Damska")])

print("\n Wszystkie miasta, rok 2024")
print(df_hi.xs(2024, level="rok"))

print("\n Kraków, kategoria Męska, Q4")
print(df_hi.xs(("Kraków", "Męska", "Q4"), level=("miasto", "kategoria", "kwartal")))


 Wszystkie dane dla Łodzi
                        Unnamed: 0  przychod_tys  koszt_tys  sztuki  marza_tys
kategoria rok  kwartal                                                        
Damska    2023 Q1                0          50.5       35.2     615       15.3
               Q2                1          59.3       34.5     539       24.8
               Q3                2          61.9       34.3     915       27.6
               Q4                3          81.8       58.6     775       23.2
          2024 Q1                4          49.5       30.2     577       19.3
               Q2                5          62.5       39.8     627       22.7
               Q3                6          53.3       32.4     563       20.9
               Q4                7          85.8       55.0    1157       30.8
Dziecięca 2023 Q1               16          20.6       14.7     216        5.9
               Q2               17          31.4       19.0     370       12.4
               Q3        

## **Zadanie 3 — pivot_table: roczne podsumowanie**

a) Utwórz tabelę przestawną `tab_miasta`, która pokaże **sumę przychodów** w wierszach per miasto, w kolumnach per rok.

b) Dodaj do tabeli kolumnę `zmiana_proc` — procentową zmianę przychodu między 2023 a 2024. Które miasto rosło najszybciej?

c) Utwórz drugą tabelę przestawną, w której wiersze to (miasto, kategoria), kolumny to rok, wartości to *średni przychód kwartalny*. Która kombinacja (miasto, kategoria) ma najwyższy średni przychód w 2024?

In [288]:
tab_miasta = df_hi.pivot_table(
    values="przychod_tys",
    index="miasto",
    columns="rok",
    aggfunc="sum"
)
print("\n Suma przychodów")
print(tab_miasta)


tab_miasta["zmiana_proc"] = (
    (tab_miasta[2024] - tab_miasta[2023]) / tab_miasta[2023] * 100
).round(2)
print("\n Zmiana procentowa")
print(tab_miasta)
najszybsze = tab_miasta["zmiana_proc"].idxmax()
print(f"Najszybciej rosnące miasto: {najszybsze} ({tab_miasta['zmiana_proc'].max():.2f}%)")


tab_miasto_kat = df_hi.pivot_table(
    values="przychod_tys",
    index=["miasto", "kategoria"],
    columns="rok",
    aggfunc="mean"
).round(2)
print("\n Średni przychód kwartalny")
print(tab_miasto_kat)
najlepsza = tab_miasto_kat[2024].idxmax()
print(f"Najwyższy średni przychód 2024: {najlepsza[0]}, {najlepsza[1]} ({tab_miasto_kat[2024].max():.2f} tys.)")


 Suma przychodów
rok       2023   2024
miasto               
Kraków   824.3  904.0
Wrocław  697.7  753.4
Łódź     570.9  637.3

 Zmiana procentowa
rok       2023   2024  zmiana_proc
miasto                            
Kraków   824.3  904.0         9.67
Wrocław  697.7  753.4         7.98
Łódź     570.9  637.3        11.63
Najszybciej rosnące miasto: Łódź (11.63%)

 Średni przychód kwartalny
rok                 2023   2024
miasto  kategoria              
Kraków  Damska     89.10  91.62
        Dziecięca  46.70  58.78
        Męska      70.28  75.60
Wrocław Damska     70.78  78.27
        Dziecięca  41.60  46.48
        Męska      62.05  63.60
Łódź    Damska     63.38  62.78
        Dziecięca  32.17  41.90
        Męska      47.18  54.65
Najwyższy średni przychód 2024: Kraków, Damska (91.62 tys.)


## **Zadanie 4 — Agregacja po poziomach**
a) Na `df_hi` oblicz sumę przychodów per miasto (agreguj po poziomie `"miasto"`).

b) Oblicz **średni przychód kwartalny per kategoria** (agreguj po poziomie `"kategoria"`).

c) Użyj `.agg(["sum", "mean", "max"])` na kolumnie `przychod_tys` pogrupowanej po `(miasto, rok)`. Który wiersz ma najwyższą wartość `max`?

In [289]:
print("Suma przychodów per miasto")
suma_miasto = df_hi["przychod_tys"].groupby(level="miasto").sum().round(2)
print(suma_miasto)


print("\n Średni przychód kwartalny per kategoria")
srednia_kategoria = df_hi["przychod_tys"].groupby(level="kategoria").mean().round(2)
print(srednia_kategoria)


print("\n Agregacja sum/mean/max per (miasto, rok)")
agg_miasto_rok = (
    df_hi["przychod_tys"]
    .groupby(level=["miasto", "rok"])
    .agg(["sum", "mean", "max"])
    .round(2)
)
print(agg_miasto_rok)

najwyzszy_wiersz = agg_miasto_rok["max"].idxmax()
najwyzszy_max    = agg_miasto_rok["max"].max()
print(f"\nWiersz z najwyższą wartością max:")
print(f"  Miasto: {najwyzszy_wiersz[0]}, Rok: {najwyzszy_wiersz[1]}, max = {najwyzszy_max} tys.")

Suma przychodów per miasto
miasto
Kraków     1728.3
Wrocław    1451.1
Łódź       1208.2
Name: przychod_tys, dtype: float64

 Średni przychód kwartalny per kategoria
kategoria
Damska       75.99
Dziecięca    44.60
Męska        62.22
Name: przychod_tys, dtype: float64

 Agregacja sum/mean/max per (miasto, rok)
                sum   mean    max
miasto  rok                      
Kraków  2023  824.3  68.69  116.9
        2024  904.0  75.33  115.3
Wrocław 2023  697.7  58.14   96.0
        2024  753.4  62.78  107.6
Łódź    2023  570.9  47.58   81.8
        2024  637.3  53.11   85.8

Wiersz z najwyższą wartością max:
  Miasto: Kraków, Rok: 2023, max = 116.9 tys.


## **Zadanie 5 — Marża i ranking**
a) W `df_hi` dodaj kolumnę `marza_tys = przychod_tys − koszt_tys`.

b) Utwórz tabelę przestawną: *wiersze = miasto*, *kolumny = kategoria*, *wartości = suma marży*. Które miasto jest najbardziej zyskowne? Która kategoria generuje najwyższą marżę?

In [290]:
df_hi["marza_tys"] = df_hi["przychod_tys"] - df_hi["koszt_tys"]

print("df_hi z kolumną marza_tys")
print(df_hi.head(8))

tab_marza = df_hi.pivot_table(
    values="marza_tys",
    index="miasto",
    columns="kategoria",
    aggfunc="sum"
).round(2)

tab_marza["SUMA"] = tab_marza.sum(axis=1).round(2)

tab_marza.loc["SUMA"] = tab_marza.sum(axis=0).round(2)

print("\n Tabela marży (miasto x kategoria)")
print(tab_marza)

najbardziej_zyskowne = tab_marza.drop("SUMA")["SUMA"].idxmax()
najlepsza_kategoria  = tab_marza.drop(index="SUMA")["SUMA"].idxmax()

print(f"\nNajbardziej zyskowne miasto:   {najbardziej_zyskowne}"
      f" ({tab_marza.loc[najbardziej_zyskowne, 'SUMA']:.2f} tys.)")

kategorie = [c for c in tab_marza.columns if c != "SUMA"]
najlepsza_kat_val = tab_marza.loc["SUMA", kategorie].idxmax()
najlepsza_kat_sum = tab_marza.loc["SUMA", najlepsza_kat_val]
print(f"Kategoria z najwyższą marżą:   {najlepsza_kat_val}"
      f" ({najlepsza_kat_sum:.2f} tys.)")

df_hi z kolumną marza_tys
                               Unnamed: 0  przychod_tys  koszt_tys  sztuki  \
miasto kategoria rok  kwartal                                                
Kraków Damska    2023 Q1               24          71.3       51.9     806   
                      Q2               25          90.8       52.1    1179   
                      Q3               26          77.4       54.5     886   
                      Q4               27         116.9       76.5    1285   
                 2024 Q1               28          73.0       44.7     843   
                      Q2               29          89.2       65.3     869   
                      Q3               30          89.0       53.0     759   
                      Q4               31         115.3       70.1    1052   

                               marza_tys  
miasto kategoria rok  kwartal             
Kraków Damska    2023 Q1            19.4  
                      Q2            38.7  
                     